<figure>
  <img src="https://raw.githubusercontent.com/shadowkshs/DimABSA2026/refs/heads/main/banner.png" width="100%">
</figure>

In [84]:
AUGMENTED = "data_augmentation_PoS_BT/output/final_adj_augment_7272.jsonl"

In [85]:
# %load_ext autoreload
# %autoreload 2

import json, yaml
from typing import List, Dict
from tqdm import tqdm
from pathlib import Path
from datetime import datetime
import logging
import sys
import os

import math
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import sentencepiece

from sklearn.model_selection import train_test_split
from scipy.stats import pearsonr

In [86]:
if "google.colab" in sys.modules :
    REPO_PATH = Path("/content/NLP_semeval26_task3_DimASR")

    if REPO_PATH.exists():
        %rm -rf "/content/NLP_semeval26_task3_DimASR"
        !git clone "https://github.com/Projet-NLP-UdeS/NLP_semeval26_task3_DimASR.git"
    else :
        !git clone "https://github.com/Projet-NLP-UdeS/NLP_semeval26_task3_DimASR.git"

    %cd "/content/NLP_semeval26_task3_DimASR"
    !git checkout colab_outputs_owen
    sys.path.insert(0, str(REPO_PATH))

from src.data import *
from src.eval import *
from src.models.svr import run_svr_baseline
from src.models.bert import TransformerVARegressor
from src.models.ensemble import (
    AverageEnsemble
)

In [87]:
log_format = "%(asctime)s | %(levelname)s | %(message)s \n"
logging.basicConfig(
    level=logging.INFO,
    format=log_format,
    force=True,
)

logger = logging.getLogger()
fh = logging.FileHandler("outputs/results/log.txt")
fh.setFormatter(logging.Formatter(log_format))
logger.addHandler(fh)

logging.info("This shows in notebook and goes to file")

2026-04-13 22:59:56,852 | INFO | This shows in notebook and goes to file 



In [88]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logging.info(f"Will be using {device} device.")
# device = torch.device("cpu") # force

# Set testing filter
# for faster training testing
testing = True # (device.type == "cpu")
if testing: (logging.info(f"Will be using a lighter training configuration, not suitable for final results."))

2026-04-13 22:59:56,874 | INFO | Will be using cpu device. 

2026-04-13 22:59:56,877 | INFO | Will be using a lighter training configuration, not suitable for final results. 



### Step 1: Load datasets and configuration


In [89]:
subtask = "subtask_1"
task = "task1"
lang = "eng"
domain = "restaurant"

!pwd
if AUGMENTED is not None : 
    logging.info("Will be using local augmented dataset")
    train_raw = load_jsonl(AUGMENTED)
else :
    logging.info("Will be using remote default dataset")
    train_url = (f"https://raw.githubusercontent.com/DimABSA/DimABSA2026/refs/heads/main/"
                 f"task-dataset/track_a/{subtask}/{lang}/{lang}_{domain}_train_alltasks.jsonl")
    train_raw = load_jsonl_url(train_url)

predict_url = (f"https://raw.githubusercontent.com/DimABSA/DimABSA2026/refs/heads/main/"
               f"task-dataset/track_a/{subtask}/{lang}/{lang}_{domain}_dev_{task}.jsonl")
predict_raw = load_jsonl_url(predict_url)

train_df = jsonl_to_df(train_raw)
predict_df = jsonl_to_df(predict_raw)

train_df = train_df.sample(100) if testing else train_df

# split 10% for dev
train_df, dev_df = train_test_split(train_df, test_size=0.1, random_state=42)


with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

models = config["models"]
logging.info(json.dumps(models, indent=2))

/home/user/UdS/IFT714/NLP_semeval26_task3_DimASR


2026-04-13 22:59:57,294 | INFO | Will be using local augmented dataset 

2026-04-13 22:59:57,563 | INFO | [
  {
    "name": "distilbert-base-uncased-finetuned-sst-2-english",
    "nickname": "baby_bert_rapid_tests",
    "type": "transformer",
    "lr": "2e-5",
    "epochs": 1,
    "batch_size": 32,
    "dropout": 0.2,
    "max_len": 64
  }
] 



### Display the dataframe

In [90]:
from IPython.display import display, Markdown

display(Markdown(f"### {subtask}_{lang}_{domain} train_df"))
display(train_df.head())

display(Markdown(f"### {subtask}_{lang}_{domain} dev_df"))
display(dev_df.head())

display(Markdown(f"### {subtask}_{lang}_{domain} predict_df"))
display(predict_df.head())

### subtask_1_eng_restaurant train_df

,Aspect,ID,Text,Valence,Arousal
8929,people,adj_aug_02749,there was a undeniably nice vibe about the pla...,7.50,7.67
11913,service,adj_aug_04303,while the ambiance and atmosphere were magnifi...,4.00,5.00
11551,restaurant,adj_aug_04110,i recently went to this restaurant with some c...,7.88,8.12
6084,waitress,adj_aug_01267,the waitress was especially patient with us an...,7.50,7.17
11036,bottles of wine,adj_aug_03828,bottles of wine are cheap and pleasant .,7.12,7.25


### subtask_1_eng_restaurant dev_df

,Aspect,ID,Text,Valence,Arousal
7058,italian restaurant,adj_aug_01772,i am so happy to have a fantastic italian rest...,7.75,7.88
11515,salad,adj_aug_04091,by far the foremost salad i have had in a fast...,7.80,7.90
11965,food,adj_aug_04331,"the food is spectacular , and the waiting staf...",7.75,7.62
8476,service,adj_aug_02508,an superb service,7.88,7.75
12058,place,adj_aug_04377,this place is remarkably much fun .,7.62,7.62


### subtask_1_eng_restaurant predict_df

,Aspect,VA,ID,Text,Valence,Arousal
0,diner food,7.25#6.75,rest26_aspect_va_dev_1,Great diner food and breakfast is served all day,7.25,6.75
1,breakfast,7.25#6.75,rest26_aspect_va_dev_1,Great diner food and breakfast is served all day,7.25,6.75
2,food,7.50#7.75,rest26_aspect_va_dev_2,It got very crowded but we still received exce...,7.50,7.75
3,drinks,7.50#7.50,rest26_aspect_va_dev_2,It got very crowded but we still received exce...,7.50,7.50
4,service,7.75#7.75,rest26_aspect_va_dev_2,It got very crowded but we still received exce...,7.75,7.75


### Step 2 : Train all models in config.yaml

In [91]:
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    checkpoint_dir = "/content/drive/MyDrive/UdS-IFT714-checkpoints"
else:
    checkpoint_dir = "outputs/checkpoints"

os.makedirs(checkpoint_dir, exist_ok=True)


def save_model_checkpoint(
    checkpoint_dir=checkpoint_dir,
    nickname="bert_model_default",
    epoch=4,
    model=None,
    optimizer=None,
    train_loss=None,
    val_loss=None,
    lr=None,
    epochs=None,
    batch_size=None,
    dropout=None,
    max_len=None
    ):
    checkpoint_path = os.path.join(
        checkpoint_dir,
        f"{nickname}_{epoch+1}_{epochs}_last.pt"
    )

    torch.save(
        {
            "model_name": nickname,
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_loss": train_loss,
            "val_loss": val_loss,
            "lr": lr,
            "batch_size": batch_size,
            "dropout": dropout,
            "max_len": max_len,
        },
        checkpoint_path,
    )

    logging.info(f"Checkpoint saved: {checkpoint_path}")

def load_model_checkpoint(
    checkpoint_dir=checkpoint_dir,
    nickname=None,
    epoch=None,
    epochs=None,
    model=None,
    optimizer=None,
    device="cpu",
    ):

    checkpoint_path = os.path.join(
        checkpoint_dir,
        f"{nickname}_{epoch+1}_{epochs}_last.pt"
    )

    checkpoint = torch.load(checkpoint_path, map_location=device)

    # # Load model weights
    # if model is not None:
    #     model.load_state_dict(checkpoint["model_state_dict"])

    # # Load optimizer state (optional)
    # if optimizer is not None and "optimizer_state_dict" in checkpoint:
    #     optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    logging.info(f"Checkpoint loaded: {checkpoint_path}")

    return checkpoint


In [92]:
model_results = {} # Pour stocker les scores finaux
trained_models = {}
ensemble = dev_df

for arch in models:
    current_model = arch["name"]
    model_type = arch["type"]
    current_nickname = arch.get("nickname", current_model)

    print(f"\n{'='*80}")
    print(f"ENTRAÎNEMENT DU MODÈLE : {current_model}")
    print(f"{'='*80}")

    # Pipline Deep learning
    if model_type == "transformer":

        current_lr = float(arch["lr"])
        current_epochs = arch["epochs"]
        current_dropout = arch["dropout"]

        current_batch_size = arch["batch_size"] if not testing else 1

        tokenizer = AutoTokenizer.from_pretrained(current_model)
        default_max_len = tokenizer.model_max_length if tokenizer.model_max_length < 1025 else 128
        current_max_len = int(arch.get("max_len", default_max_len))
        logging.info(f"Current maximum token length is {current_max_len}")

        print(f"Paramètres : LR={current_lr}, Epochs={current_epochs}, Batch={current_batch_size}, Dropout={current_dropout}")

        # Création des DataLoaders
        train_dataset = VADataset(train_df, tokenizer, max_len=current_max_len)
        dev_dataset = VADataset(dev_df, tokenizer, max_len=current_max_len)

        train_loader = DataLoader(train_dataset, batch_size=current_batch_size, shuffle=True)
        dev_loader = DataLoader(dev_dataset, batch_size=current_batch_size, shuffle=False)

        # Initialisation du modèle
        model = TransformerVARegressor(current_model_name=current_model, dropout=current_dropout).to(device).float()
        optimizer = torch.optim.AdamW(model.parameters(), lr=current_lr)
        loss_fn = nn.MSELoss()

        # Entraînement du modèle
        for epoch in range(current_epochs):
            train_loss = model.train_epoch(train_loader, optimizer, loss_fn, device)
            val_loss = model.eval_epoch(dev_loader, loss_fn, device)
            logging.info(f"Epoch {epoch+1}/{current_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

            save_model_checkpoint(
                checkpoint_dir=checkpoint_dir,
                nickname=current_nickname,
                model=model,
                optimizer=optimizer,
                epoch=epoch,
                train_loss=train_loss,
                val_loss=val_loss,
                lr=current_lr,
                epochs=current_epochs,
                batch_size=current_batch_size,
                dropout=current_dropout,
                max_len=current_max_len
            )

        # Évaluation du modèle sur le Dev Set
        pred_v, pred_a, gold_v, gold_a = get_prd(model, dev_loader, type="dev")
        eval_score = evaluate_predictions_task1(pred_a, pred_v, gold_a, gold_v)
        model_results[current_nickname] = eval_score
        trained_models[current_nickname] = model

        # Saving predictions for ensemble learning
        ensemble = predict_to_dataframe(
            model, dev_loader, ensemble,
            pred_v_col = f"{current_nickname}_valence",
            pred_a_col = f"{current_nickname}_arousal"
        )

    # Pipline Machine Learning
    elif model_type == "sklearn":

        max_features = arch["max_features"]

        pred_v, pred_a, gold_v, gold_a = run_svr_baseline(train_df, dev_df, max_features=max_features)

        eval_score = evaluate_predictions_task1(pred_a, pred_v, gold_a, gold_v)
        model_results[current_nickname] = eval_score

        # ensemble = predict_to_dataframe(
        #     model, dev_loader, ensemble,
        #     pred_v_col = f"{current_model}_valence",
        #     pred_a_col = f"{current_model}_arousal"
        # )


ENTRAÎNEMENT DU MODÈLE : distilbert-base-uncased-finetuned-sst-2-english


2026-04-13 22:59:57,930 | INFO | HTTP Request: HEAD https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english/resolve/main/config.json "HTTP/1.1 200 OK" 

2026-04-13 22:59:57,984 | INFO | HTTP Request: HEAD https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK" 

2026-04-13 22:59:58,062 | INFO | HTTP Request: GET https://huggingface.co/api/models/distilbert-base-uncased-finetuned-sst-2-english/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect" 

2026-04-13 22:59:58,109 | INFO | HTTP Request: GET https://huggingface.co/api/models/distilbert/distilbert-base-uncased-finetuned-sst-2-english/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found" 

2026-04-13 22:59:58,152 | INFO | HTTP Request: GET https://huggingface.co/api/models/distilbert-base-uncased-finetuned-sst-2-english/tree/main?recursive=true&expand=false "HTTP/1.1 307

Paramètres : LR=2e-05, Epochs=1, Batch=1, Dropout=0.2


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased-finetuned-sst-2-english
Key                   | Status     |  | 
----------------------+------------+--+-
pre_classifier.weight | UNEXPECTED |  | 
pre_classifier.bias   | UNEXPECTED |  | 
classifier.bias       | UNEXPECTED |  | 
classifier.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-04-13 23:00:42,349 | INFO | Epoch 1/1 | Train Loss: 16.1045 | Val Loss: 3.6884 

2026-04-13 23:00:45,051 | INFO | Checkpoint saved: outputs/checkpoints/baby_bert_rapid_tests_1_1_last.pt 



In [93]:
# torch.save(model.state_dict(), "outputs/checkpoints/distillbert_test.pt")
# temp to avoid retraining, needs to be put into training function

path = "outputs/results/preds.csv"
ensemble.to_csv(path)

ensemble_model = AverageEnsemble(path)
pred_v, pred_a, gold_v, gold_a = ensemble_model.predictions()

eval_score = evaluate_predictions_task1(pred_a, pred_v, gold_a, gold_v)
model_results["average_ensemble"] = eval_score
trained_models["average_ensemble"] = ensemble_model


### Step 3 : Analyze results

In [94]:
with open("./outputs/results/metrics.yaml", "w") as f:
    # yaml.safe_dump(model_results, f)
    pass

In [95]:

logging.info("Récapitulatif des résultats:")
if "google.colab" in sys.modules :
    logging.info(f"Running in Colab with {device} device...")
for mod, scores in model_results.items():
    line = (
        f"- {mod} : "
        f"PCC_V = {scores['PCC_V']:.4f} | "
        f"PCC_A = {scores['PCC_A']:.4f} | "
        f"RMSE_V = {scores['RMSE_V']:.4f} | "
        f"RMSE_A = {scores['RMSE_A']:.4f}| "
        f"RMSE_VA = {scores['RMSE_VA']:.4f}"
    )
    logging.info(line)

2026-04-13 23:00:48,540 | INFO | Récapitulatif des résultats: 

2026-04-13 23:00:48,542 | INFO | - baby_bert_rapid_tests : PCC_V = 0.1397 | PCC_A = -0.5091 | RMSE_V = 1.9808 | RMSE_A = 1.8584| RMSE_VA = 1.9205 

2026-04-13 23:00:48,545 | INFO | - average_ensemble : PCC_V = 0.1397 | PCC_A = -0.5091 | RMSE_V = 1.9808 | RMSE_A = 1.8584| RMSE_VA = 1.9205 



In [96]:
# CTRL+S to commit main.ipynb and...
# but doesn't work anymore in organization repo...
if "google.colab" in sys.modules :
  from google.colab import userdata, _message
  from getpass import getpass

  try :
    resp = _message.blocking_request('get_ipynb', timeout_sec=5)
    if not resp or not isinstance(resp, dict):
        raise ValueError("Couldn't fetch Colab notebook to commit.")
    with open('main.ipynb', 'w') as f:
        json.dump(resp['ipynb'], f)
  except Exception as e:
     print(type(e).__name__, "-", e)

  # GitHub / Settings / Emails (look for 123+user@users.noreply.github.com)
  try:
    email = userdata.get("GITHUB_EMAIL")
  except Exception:
    email = input("Enter your email: ")
  !git config --global user.email {email}

  try:
    name = userdata.get("GITHUB_NAME")
  except Exception:
    name = input("Enter your email: ")
  !git config --global user.name {name}

  !git status
  print()

  !git add outputs/ main.ipynb
  !git commit -m "feat: auto colab outputs"
  print()

  # GitHub / Settings / Developer settings / Personal access tokens
  try:
    token = userdata.get("GITHUB_TOKEN")
  except Exception:
    token = getpass("Enter GitHub token: ")
  !git push "https://{token}@github.com/Projet-NLP-UdeS/NLP_semeval26_task3_DimASR.git"